# Train Faster R-CNN ResNet-50-FPN trên Kaggle

Notebook one-click: bật **GPU** và **Internet**, Add Input processed VisDrone và raw `VisDrone2019-DET-val`, rồi chọn **Run All**. Từ F5, mỗi epoch được đánh giá bằng VisDrone DET metric và `best.pth` được chọn theo `visdrone_ap`.

- `F0`: baseline pretrained COCO, không augmentation.
- `F1`: giữ nguyên cấu hình F0, chỉ thêm augmentation bbox-safe.
- `F2`: giữ nguyên F0, chỉ thay anchor FPN thành `8,16,32,64,128` cho vật thể nhỏ.
- `F3`: giữ nguyên F0, chỉ tăng RPN proposals cho ảnh crowded.
- `F4`: giữ proposals của F3, đổi sang Faster R-CNN ResNet-50-FPN V2 pretrained COCO.
- `F5`: giữ F4, tăng input resize từ `800/1333` lên `1024/1707`.
- Smoke test chạy 2 batch để kiểm tra pipeline; kết quả không dùng để báo cáo.


## 1. Cấu hình

F5 cần processed dataset để train và raw validation để giữ ignore regions khi đánh giá. Khi resume, chỉ dùng `last.pth` của chính F5.


In [ ]:
from pathlib import Path

EXPERIMENT = "F5"           # F5 = F4 + input resolution 1024/1707
RUN_SMOKE_FIRST = True        # tự kiểm tra 2 batch trước full training
TRAIN_FULL = True             # train đủ EPOCHS ngay sau smoke test
EPOCHS = 25
BATCH_SIZE = 1 if EXPERIMENT in {"F4", "F5"} else 2
MIN_SIZE = 1024 if EXPERIMENT == "F5" else 800
MAX_SIZE = 1707 if EXPERIMENT == "F5" else 1333
WORKERS = 2
SEED = 42

REPO_URL = "https://github.com/Nhattk19/Object_dectection.git"
REPO_BRANCH = "cnn-faster-rcnn-pipeline"
PROJECT_ROOT_OVERRIDE = None  # thường không cần sửa
PROCESSED_DATA_ROOT = None    # tự tìm trong /kaggle/input; có thể điền thủ công
RAW_VAL_ROOT = None           # raw VisDrone2019-DET-val có images/ + annotations/
RESUME_FROM = None            # ví dụ: /kaggle/input/f0-checkpoint/f0/last.pth
OUTPUT_ROOT = Path("/kaggle/working/faster_rcnn_runs")

assert EXPERIMENT in {"F0", "F1", "F2", "F3", "F4", "F5"}


## 2. Clone repository và kiểm tra GPU


In [ ]:
import importlib.util
import subprocess
import sys

def find_project_root():
    if PROJECT_ROOT_OVERRIDE:
        candidate = Path(PROJECT_ROOT_OVERRIDE)
        if (candidate / "src" / "faster_rcnn.py").is_file():
            return candidate.resolve()
        raise FileNotFoundError(f"PROJECT_ROOT_OVERRIDE không hợp lệ: {candidate}")
    direct = [Path.cwd(), *Path.cwd().parents, Path("/kaggle/working")]
    for candidate in direct:
        if (candidate / "src" / "faster_rcnn.py").is_file():
            return candidate.resolve()
    for base in (Path("/kaggle/working"), Path("/kaggle/input")):
        if base.exists():
            matches = list(base.glob("**/src/faster_rcnn.py"))
            if matches:
                return matches[0].parents[1].resolve()
    clone_dir = Path("/kaggle/working/Object_dectection")
    subprocess.check_call(["git", "clone", "--depth", "1", "--branch",
                           REPO_BRANCH, REPO_URL, str(clone_dir)])
    return clone_dir.resolve()

PROJECT_ROOT = find_project_root()
if importlib.util.find_spec("pycocotools") is None:
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", "pycocotools>=2.0.7"])

import torch, torchvision
assert torch.cuda.is_available(), "Chưa có CUDA. Vào Kaggle Settings và bật GPU Accelerator."
print({"project": str(PROJECT_ROOT), "torch": torch.__version__,
       "torchvision": torchvision.__version__, "gpu": torch.cuda.get_device_name(0)})


## 3. Tìm và kiểm tra processed COCO dataset

Dataset đã Add Input phải chứa `images/train`, `images/val`, `annotations/instances_train.json` và `instances_val.json`. Upload processed data giúp không phải chuyển đổi lại trong mỗi Kaggle session.


In [ ]:
def valid_processed_root(path):
    path = Path(path)
    return all((path / item).exists() for item in (
        "images", "annotations/instances_train.json", "annotations/instances_val.json"
    ))

if PROCESSED_DATA_ROOT:
    DATA_ROOT = Path(PROCESSED_DATA_ROOT)
    if not valid_processed_root(DATA_ROOT):
        raise FileNotFoundError(f"Processed dataset không hợp lệ: {DATA_ROOT}")
else:
    candidates = []
    for base in (Path("/kaggle/input"), PROJECT_ROOT / "data"):
        if base.exists():
            candidates.extend(p.parent.parent for p in base.glob("**/annotations/instances_train.json"))
    DATA_ROOT = next((p for p in candidates if valid_processed_root(p)), None)

if DATA_ROOT is None:
    raise FileNotFoundError(
        "Không tìm thấy processed VisDrone trong Kaggle Input. Hãy upload/add thư mục "
        "data/processed/VisDrone hoặc đặt PROCESSED_DATA_ROOT thủ công."
    )

assert valid_processed_root(DATA_ROOT)
import json
train_payload = json.loads((DATA_ROOT / "annotations/instances_train.json").read_text())
val_payload = json.loads((DATA_ROOT / "annotations/instances_val.json").read_text())
def valid_raw_val(path):
    path = Path(path)
    return (path / "images").is_dir() and len(list((path / "annotations").glob("*.txt"))) >= 500

if RAW_VAL_ROOT:
    RAW_ROOT = Path(RAW_VAL_ROOT)
else:
    RAW_ROOT = next((p.parent for p in Path("/kaggle/input").glob("**/annotations")
                     if valid_raw_val(p.parent)), None)
if EXPERIMENT == "F5" and (RAW_ROOT is None or not valid_raw_val(RAW_ROOT)):
    raise FileNotFoundError("F5 cần Add Input raw VisDrone2019-DET-val để evaluate toolkit")

print({"processed_root": str(DATA_ROOT), "raw_val_root": str(RAW_ROOT),
       "train_images": len(train_payload["images"]),
       "train_boxes": len(train_payload["annotations"]),
       "val_images": len(val_payload["images"]),
       "val_boxes": len(val_payload["annotations"])})


## 4. Preview ground truth và augmentation

Ảnh bên trái là dữ liệu gốc; bên phải là augmentation bbox-safe của F1. F0 không dùng augmentation khi train.


In [ ]:
sys.path.insert(0, str(PROJECT_ROOT / "src"))
import matplotlib.pyplot as plt
from matplotlib.patches import Rectangle
from augmentation import ControlledDetectionAugmenter
from faster_rcnn import VISDRONE_CLASS_NAMES, VisDroneCocoDataset

TRAIN_JSON = DATA_ROOT / "annotations/instances_train.json"
IMAGE_ROOT = DATA_ROOT / "images"
plain_dataset = VisDroneCocoDataset(TRAIN_JSON, IMAGE_ROOT)
aug_dataset = VisDroneCocoDataset(
    TRAIN_JSON, IMAGE_ROOT, augmentation=ControlledDetectionAugmenter(seed=SEED)
)

def draw_sample(ax, image, target, title, max_boxes=150):
    ax.imshow(image.permute(1, 2, 0).numpy())
    for box, label in zip(target["boxes"][:max_boxes], target["labels"][:max_boxes]):
        x1, y1, x2, y2 = box.tolist()
        ax.add_patch(Rectangle((x1, y1), x2-x1, y2-y1, fill=False,
                               edgecolor=plt.cm.tab10(int(label) % 10), linewidth=.7))
    ax.set_title(f"{title}: {len(target['boxes'])} boxes")
    ax.axis("off")

plain_image, plain_target = plain_dataset[0]
aug_image, aug_target = aug_dataset[0]
fig, axes = plt.subplots(1, 2, figsize=(16, 6))
draw_sample(axes[0], plain_image, plain_target, "Ground truth")
draw_sample(axes[1], aug_image, aug_target, "F1 augmentation preview")
plt.tight_layout(); plt.show()


## 5. Smoke test rồi train full

Notebook tự chạy smoke test trong thư mục riêng, sau đó khởi tạo lại model pretrained COCO để train full. Output gồm `last.pth`, `best.pth`, `history.csv`, `learning_curves.png`, `config.json` và `summary.json`.


In [ ]:
def run_training(output_root, *, smoke=False, resume_from=None):
    command = [
        sys.executable, "-u", str(PROJECT_ROOT / "scripts/train_faster_rcnn.py"),
        "--data-root", str(DATA_ROOT), "--output-dir", str(output_root),
        "--experiment", EXPERIMENT, "--epochs", str(EPOCHS),
        "--batch-size", str(BATCH_SIZE), "--workers", str(WORKERS),
        "--seed", str(SEED), "--min-size", str(MIN_SIZE),
        "--max-size", str(MAX_SIZE),
    ]
    if RAW_ROOT:
        command.extend(["--raw-val-root", str(RAW_ROOT)])
    if smoke:
        command.append("--smoke-test")
    if resume_from:
        command.extend(["--resume", str(resume_from)])
    print("Running:", " ".join(command))
    subprocess.check_call(command, cwd=PROJECT_ROOT)

if RUN_SMOKE_FIRST and not RESUME_FROM:
    run_training(Path("/kaggle/working/smoke_runs"), smoke=True)
if TRAIN_FULL:
    run_training(OUTPUT_ROOT, resume_from=RESUME_FROM)


## 6. Xem kết quả và tạo Kaggle Output


In [ ]:
import json
import pandas as pd
from IPython.display import Image as DisplayImage, display

RUN_DIR = OUTPUT_ROOT / EXPERIMENT.lower()
summary = json.loads((RUN_DIR / "summary.json").read_text())
display(pd.DataFrame([summary]))
display(pd.read_csv(RUN_DIR / "history.csv"))
if (RUN_DIR / "learning_curves.png").is_file():
    display(DisplayImage(filename=str(RUN_DIR / "learning_curves.png")))
print("Artifacts để Save Version / tải về:", RUN_DIR)
